# Loop0, stage 01b: read the fluorescence-minus-one controls

A fluorescence-minus-one control is a sample stained with every antibody of the
panel except one.  Any signal that appears in the missing antibody's channel is
therefore background rather than staining, which is what stage 03 measures.

One well per control, plus one fully stained well that is dropped here because
the fully stained cells come from the screen itself in stage 03.

## Inputs

| file | produced by | holds |
|---|---|---|
| `$LABCOMPASS_FCS_ROOT/LPFMO02` | acquisition, outside the repository | one control per well |

## Outputs

| file | holds |
|---|---|
| `Loop0_FMOs.h5ad` | every control cell, labelled with the antibody it is missing |

Run the cells from top to bottom.

In [ ]:
import gc
import glob
import os

import anndata as ann
import numpy as np
import pandas as pd
import pytometry as pm

# The control plate, one well per antibody left out.
FCS_ROOT = os.environ.get("LABCOMPASS_FCS_ROOT", "/rds/project/rds-SDzz0CATGms/unsorted/for_Juan_Licyel")
FMO_DATA_PATH = os.path.join(FCS_ROOT, "LPFMO02")

OUT_FMO = "Loop0_FMOs.h5ad"

LIVE_DEAD_CHANNEL = "Live_Dead : 7-AAD - Area"
CHANNELS_TO_DROP = [
    "CD184 : PerCP-eFluor710 - Area",
    LIVE_DEAD_CHANNEL,
    "[AF color 1] - Area",
    "[AF color 2] - Area",
    "[AF color 3] - Area",
    "[AF color 4] - Area",
    "[AF color 5] - Area",
    "[AF color 6] - Area",
    "[AF color 7] - Area",
]

## The control wells

The last file of the plate is the fully stained control, not a
fluorescence-minus-one control, so it is dropped.

In [ ]:
files = sorted(
    glob.glob(os.path.join(FMO_DATA_PATH, "**", "filtered", "*AutoSpectral.fcs"),
              recursive=True)
)
files = files[:-1]
print(f"{len(files)} control wells")
files

## Read each control and label it with the antibody it is missing

The antibody name is taken from the file name.  A file called
`A01 -CD49c AutoSpectral.fcs` is the control that leaves out CD49c.

In [ ]:
def fmo_name_from_path(path):
    """Return the antibody that this control leaves out, from its file name."""
    name = path.split(" ", 1)[1] if " " in path else path
    name = name.replace("-", "").replace(" AutoSpectral.fcs", "").strip()
    if name.startswith("Fully"):
        return name.replace(" ", "_").lower()
    return name


adatas = []
for path in files:
    adata = pm.io.read_fcs(path)
    adata.obs.index = adata.obs.index.astype(str)
    adata.var_names = adata.var["marker"].astype(str)
    if adata.n_obs == 0:
        continue

    adata.obs["fmo"] = fmo_name_from_path(path)
    adata.obs["replicate"] = 1
    adata.obs["date"] = adata.uns["meta"]["date"]
    adata.obs["cytometer"] = adata.uns["meta"]["cyt"]
    adata.obs["cytometer_serial_no"] = adata.uns["meta"]["cytsn"]
    adata.obs["well_id"] = adata.uns["meta"]["smno"]
    adata.obs["cell_counts"] = adata.n_obs

    pm.pp.split_signal(adata, var_key="channel", option="area", data_type="facs")
    adatas.append(adata)

del adata
gc.collect()
print(f"{len(adatas)} controls read")
print(sorted({a.obs["fmo"].iloc[0] for a in adatas}))

## Concatenate and match the channels of the screen

In [ ]:
adata_all = ann.concat(
    adatas, join="inner", merge="same", uns_merge="unique",
    label="batch", index_unique="-",
)
del adatas
gc.collect()

# Same three steps as stage 00, so the controls carry the same channels.
values = adata_all[:, LIVE_DEAD_CHANNEL].X
adata_all.obs["7-AAD"] = np.asarray(
    values.toarray() if hasattr(values, "toarray") else values
).ravel()

adata_all = adata_all[:, ~np.isin(adata_all.var["marker"].values, CHANNELS_TO_DROP)]

antibodies = [m.split(" :")[0] for m in adata_all.var["marker"].values]
adata_all.var = adata_all.var.copy()
adata_all.var["antibody"] = pd.Categorical(antibodies)
adata_all.var_names = antibodies

adata_all.write(OUT_FMO)
print(f"wrote {OUT_FMO}: {adata_all.n_obs:,} cells x {adata_all.n_vars} antibodies")
adata_all